# GA4 Exploratory Data Analysis

**Google Merchandise Store · Nov 2020 – Jan 2021 · ~360K sessions · ~4.3M events**

Goal: understand data structure, quality, and distributions before A/A test validation and funnel evaluation.

> **Prerequisites:** `dbt build --select staging+` must complete before running cells that query intermediate/mart tables.

## 0. Setup

Imports, BigQuery client, shared config.

In [ ]:
# Install required packages if not already present
# Run this cell once, then restart the kernel
%pip install --quiet google-cloud-bigquery[pandas] db-dtypes scipy seaborn matplotlib pandas numpy

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from google.cloud import bigquery

# ── BigQuery table paths ─────────────────────────────────────────
PROJECT = 'ga4-bigquery-abtest'
STG     = f'{PROJECT}.dbt_ga4_funnel_staging.stg_ga4__events'
INT     = f'{PROJECT}.dbt_ga4_funnel_intermediate'
MART    = f'{PROJECT}.dbt_ga4_funnel_marts'
RAW     = 'bigquery-public-data.ga4_obfuscated_sample_ecommerce'
D0, D1  = '20201101', '20210131'

# ── Funnel step labels ───────────────────────────────────────────
FUNNEL_COLS   = ['n_session_start', 'n_view_item', 'n_add_to_cart',
                 'n_begin_checkout', 'n_purchase']
FUNNEL_LABELS = ['Session Start', 'View Item', 'Add to Cart',
                 'Begin Checkout', 'Purchase']
FUNNEL_EVENTS = ['session_start', 'view_item', 'add_to_cart',
                 'begin_checkout', 'purchase']
C_CTRL, C_TRTM = '#4A90D9', '#E67E22'

# ── Plot style ───────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

client = bigquery.Client(project=PROJECT)

def bq(sql):
    """Run BigQuery SQL and return a pandas DataFrame."""
    return client.query(sql).to_dataframe()

print('Setup complete ✓')

## 1. Data Overview

Shape, date range, and unique entity counts from `stg_ga4__events`.

In [ ]:
overview = bq(f'''
SELECT
  COUNT(*)                                                              AS total_events,
  COUNT(DISTINCT user_pseudo_id)                                        AS unique_users,
  COUNT(DISTINCT CONCAT(user_pseudo_id, CAST(session_id AS STRING)))    AS unique_sessions,
  COUNT(DISTINCT event_name)                                            AS event_types,
  MIN(event_date)                                                       AS first_date,
  MAX(event_date)                                                       AS last_date,
  COUNT(DISTINCT IF(event_name = 'purchase', transaction_id, NULL))     AS transactions,
  ROUND(SUM(IF(event_name = 'purchase', revenue_usd, 0)), 2)            AS total_revenue_usd
FROM `{STG}`
''')

display(overview.T.rename(columns={0: 'value'}).astype(str))

In [ ]:
# First 5 rows — shows the flattened event schema after dbt staging
sample = bq(f'SELECT * FROM `{STG}` LIMIT 5')
display(sample)

## 2. Data Quality

Null coverage per column and daily event volume to check for gaps in the data.

In [ ]:
null_df = bq(f'''
SELECT
  ROUND(COUNTIF(session_id           IS NULL) / COUNT(*) * 100, 1) AS session_id,
  ROUND(COUNTIF(page_location        IS NULL) / COUNT(*) * 100, 1) AS page_location,
  ROUND(COUNTIF(page_title           IS NULL) / COUNT(*) * 100, 1) AS page_title,
  ROUND(COUNTIF(engagement_time_msec IS NULL) / COUNT(*) * 100, 1) AS engagement_time_msec,
  ROUND(COUNTIF(session_engaged      IS NULL) / COUNT(*) * 100, 1) AS session_engaged,
  ROUND(COUNTIF(transaction_id       IS NULL) / COUNT(*) * 100, 1) AS transaction_id,
  ROUND(COUNTIF(revenue_usd          IS NULL) / COUNT(*) * 100, 1) AS revenue_usd,
  ROUND(COUNTIF(traffic_source       IS NULL) / COUNT(*) * 100, 1) AS traffic_source,
  ROUND(COUNTIF(traffic_medium       IS NULL) / COUNT(*) * 100, 1) AS traffic_medium,
  ROUND(COUNTIF(device_category      IS NULL) / COUNT(*) * 100, 1) AS device_category,
  ROUND(COUNTIF(country              IS NULL) / COUNT(*) * 100, 1) AS country,
  ROUND(COUNTIF(city                 IS NULL) / COUNT(*) * 100, 1) AS city
FROM `{STG}`
''').T.rename(columns={0: 'null_pct'}).sort_values('null_pct', ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
null_df['null_pct'].plot.barh(ax=ax, color='#E67E22', edgecolor='white')
ax.set_title('Null % by Column')
ax.set_xlabel('% null rows')
for bar, val in zip(ax.patches, null_df['null_pct']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.set_xlim(0, 105)
plt.tight_layout()
plt.show()

In [ ]:
daily_vol = bq(f'''
SELECT PARSE_DATE('%Y%m%d', event_date) AS date, COUNT(*) AS events
FROM `{STG}`
GROUP BY 1 ORDER BY 1
''')

fig, ax = plt.subplots(figsize=(13, 3))
ax.bar(daily_vol['date'], daily_vol['events'],
       color='#4A90D9', width=1, edgecolor='none')
ax.set_title('Daily Event Volume — no gaps means complete coverage')
ax.set_ylabel('Events')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Event Distribution

Which event types dominate the log? Funnel events are highlighted in red.

In [ ]:
events_df = bq(f'''
SELECT event_name, COUNT(*) AS cnt
FROM `{STG}`
GROUP BY 1 ORDER BY 2 DESC
LIMIT 20
''')

palette = ['#E63946' if e in FUNNEL_EVENTS else '#4A90D9'
           for e in events_df['event_name']]

fig, ax = plt.subplots(figsize=(9, 7))
events_df.sort_values('cnt').plot.barh(
    x='event_name', y='cnt', ax=ax,
    color=palette[::-1], legend=False
)
ax.set_title('Top 20 Events  (red = funnel events)')
ax.set_xlabel('Event count')
ax.yaxis.set_tick_params(labelsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

## 4. Temporal Patterns

Daily traffic, day-of-week patterns, and hour-of-day profile (all timestamps in UTC).

In [ ]:
daily_sess = bq(f'''
SELECT
  PARSE_DATE('%Y%m%d', event_date)                                     AS date,
  COUNT(DISTINCT CONCAT(user_pseudo_id, CAST(session_id AS STRING)))   AS sessions,
  COUNTIF(event_name = 'purchase')                                      AS purchases
FROM `{STG}`
WHERE session_id IS NOT NULL
GROUP BY 1 ORDER BY 1
''')

fig, ax1 = plt.subplots(figsize=(13, 4))
ax2 = ax1.twinx()
ax1.plot(daily_sess['date'], daily_sess['sessions'],
         color=C_CTRL, lw=2, label='Sessions')
ax2.plot(daily_sess['date'], daily_sess['purchases'],
         color=C_TRTM, lw=2, linestyle='--', label='Purchases')
ax1.set_title('Daily Sessions and Purchases — Nov 2020 to Jan 2021')
ax1.set_ylabel('Sessions', color=C_CTRL)
ax2.set_ylabel('Purchases', color=C_TRTM)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45, ha='right')
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
dow = bq(f'''
SELECT
  EXTRACT(DAYOFWEEK FROM PARSE_DATE('%Y%m%d', event_date)) AS dow_num,
  FORMAT_DATE('%A', PARSE_DATE('%Y%m%d', event_date))      AS day_name,
  COUNT(*) AS events
FROM `{STG}`
GROUP BY 1, 2 ORDER BY 1
''')

hourly = bq(f'''
SELECT EXTRACT(HOUR FROM event_timestamp) AS hour, COUNT(*) AS events
FROM `{STG}`
GROUP BY 1 ORDER BY 1
''')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(data=dow, x='day_name', y='events', ax=ax1, color=C_CTRL)
ax1.set_title('Events by Day of Week')
ax1.set_xlabel('')
ax1.set_ylabel('Event count')
ax1.tick_params(axis='x', rotation=30)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2.bar(hourly['hour'], hourly['events'], color=C_CTRL, edgecolor='none')
ax2.set_title('Events by Hour of Day (UTC)')
ax2.set_xlabel('Hour (UTC)')
ax2.set_ylabel('Event count')
ax2.set_xticks(range(0, 24, 2))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

## 5. User & Session Behaviour

Sessions-per-user distribution, session length, pageviews, and engagement rate.

In [ ]:
sess_per_user = bq(f'''
SELECT sessions, COUNT(*) AS users
FROM (
  SELECT user_pseudo_id, COUNT(DISTINCT session_id) AS sessions
  FROM `{STG}`
  WHERE session_id IS NOT NULL
  GROUP BY 1
)
GROUP BY 1 ORDER BY 1
LIMIT 30
''')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(sess_per_user['sessions'], sess_per_user['users'],
       color=C_CTRL, width=0.8, edgecolor='none')
ax.set_yscale('log')
ax.set_title('Sessions per User (log scale, capped at 30)')
ax.set_xlabel('Number of sessions')
ax.set_ylabel('Number of users (log scale)')
plt.tight_layout()
plt.show()

In [ ]:
# Requires int_sessions to be materialised by dbt
duration = bq(f'''
SELECT
  LEAST(CAST(FLOOR(session_duration_seconds / 60) AS INT64), 60) AS duration_min,
  COUNT(*) AS sessions
FROM `{INT}.int_sessions`
GROUP BY 1 ORDER BY 1
''')

pv_per_sess = bq(f'''
SELECT
  LEAST(pageviews, 20) AS pageviews_bucket,
  COUNT(*) AS sessions
FROM `{INT}.int_sessions`
GROUP BY 1 ORDER BY 1
''')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.bar(duration['duration_min'], duration['sessions'],
        color=C_CTRL, width=0.9, edgecolor='none')
ax1.set_title('Session Duration (capped at 60 min)')
ax1.set_xlabel('Duration (minutes)')
ax1.set_ylabel('Sessions')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

ax2.bar(pv_per_sess['pageviews_bucket'], pv_per_sess['sessions'],
        color='#2ECC71', width=0.9, edgecolor='none')
ax2.set_title('Pageviews per Session (capped at 20)')
ax2.set_xlabel('Pageviews')
ax2.set_ylabel('Sessions')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

In [ ]:
engaged = bq(f'''
SELECT is_engaged, COUNT(*) AS sessions
FROM `{INT}.int_sessions`
GROUP BY 1 ORDER BY 1
''')

vals    = engaged.sort_values('is_engaged')['sessions'].tolist()
labels  = ['Not Engaged', 'Engaged']

fig, ax = plt.subplots(figsize=(5, 5))
ax.pie(vals, labels=labels, autopct='%1.1f%%',
       colors=['#ADB5BD', C_CTRL], startangle=90,
       wedgeprops={'edgecolor': 'white', 'linewidth': 2})
ax.set_title('Session Engagement Rate')
plt.tight_layout()
plt.show()

print(f'Engaged sessions: {vals[1]:,} / {sum(vals):,}  ({vals[1]/sum(vals)*100:.1f}%)')

## 6. Acquisition — Traffic Sources

Where do users come from? Top sources and mediums by session count.

In [ ]:
sources = bq(f'''
SELECT
  COALESCE(NULLIF(traffic_source, ''), '(not set)') AS source,
  COALESCE(NULLIF(traffic_medium, ''), '(not set)') AS medium,
  COUNT(*) AS sessions
FROM `{INT}.int_sessions`
GROUP BY 1, 2
ORDER BY sessions DESC
LIMIT 30
''')

top_src = (sources.groupby('source')['sessions'].sum()
           .sort_values(ascending=True).tail(10))
top_med = (sources.groupby('medium')['sessions'].sum()
           .sort_values(ascending=True))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

top_src.plot.barh(ax=ax1, color=C_CTRL)
ax1.set_title('Top 10 Traffic Sources')
ax1.set_xlabel('Sessions')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

top_med.plot.barh(ax=ax2, color=C_TRTM)
ax2.set_title('Traffic Medium Breakdown')
ax2.set_xlabel('Sessions')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

## 7. Device & Geo

Device mix, geographic spread, and how conversion rate varies by device.

In [ ]:
device_df = bq(f'''
SELECT device_category, COUNT(*) AS sessions
FROM `{INT}.int_sessions`
GROUP BY 1 ORDER BY 2 DESC
''')

geo_df = bq(f'''
SELECT country, COUNT(*) AS sessions
FROM `{INT}.int_sessions`
WHERE country IS NOT NULL AND country NOT IN ('(not set)', '')
GROUP BY 1 ORDER BY 2 DESC
LIMIT 15
''')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(device_df['device_category'], device_df['sessions'],
        color=['#4A90D9', '#E67E22', '#2ECC71'], edgecolor='white')
ax1.set_title('Sessions by Device Category')
ax1.set_xlabel('')
ax1.set_ylabel('Sessions')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(ax1.patches, device_df['sessions'].tolist()):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
             f'{val:,}', ha='center', fontsize=10)

geo_df.sort_values('sessions', ascending=True).plot.barh(
    x='country', y='sessions', ax=ax2,
    color='#4A90D9', legend=False)
ax2.set_title('Top 15 Countries by Sessions')
ax2.set_xlabel('Sessions')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

In [ ]:
funnel_mart = bq(f'SELECT * FROM `{MART}.mart_funnel_metrics`')

device_conv = (
    funnel_mart.groupby('device_category')
    .agg(sessions=('total_sessions', 'sum'), purchases=('n_purchase', 'sum'))
    .reset_index()
)
device_conv['conv_pct'] = device_conv['purchases'] / device_conv['sessions'] * 100

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(device_conv['device_category'], device_conv['conv_pct'],
              color=['#4A90D9', '#E67E22', '#2ECC71'], edgecolor='white')
ax.set_title('Purchase Conversion Rate by Device')
ax.set_ylabel('Conversion rate (%)')
for bar, val in zip(bars, device_conv['conv_pct'].tolist()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.2f}%', ha='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print(device_conv[['device_category', 'sessions', 'purchases', 'conv_pct']].to_string(index=False))

## 8. Products & Categories

Revenue, purchase volume, and price distributions across the product catalogue.
Uses raw `events_*` with `UNNEST(items)` for item-level data.

In [ ]:
cat_rev = bq(f'''
SELECT
  item_category,
  COUNT(DISTINCT item_name)            AS distinct_products,
  ROUND(SUM(item_revenue_in_usd), 0)   AS revenue_usd,
  ROUND(AVG(price_in_usd), 2)          AS avg_price_usd,
  COUNT(*)                              AS items_sold
FROM `{RAW}.events_*`,
UNNEST(items)
WHERE _TABLE_SUFFIX BETWEEN '{D0}' AND '{D1}'
  AND event_name = 'purchase'
  AND item_category NOT IN ('(not set)', 'Uncategorized Items', '')
GROUP BY 1 ORDER BY 3 DESC
LIMIT 12
''')

top_prod = bq(f'''
SELECT
  item_name,
  item_category,
  COUNT(*)                AS purchases,
  ROUND(AVG(price_in_usd), 2) AS avg_price_usd
FROM `{RAW}.events_*`,
UNNEST(items)
WHERE _TABLE_SUFFIX BETWEEN '{D0}' AND '{D1}'
  AND event_name = 'purchase'
  AND item_name NOT IN ('(not set)', '')
GROUP BY 1, 2 ORDER BY 3 DESC
LIMIT 15
''')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

cat_rev.sort_values('revenue_usd', ascending=True).plot.barh(
    x='item_category', y='revenue_usd', ax=ax1, color='#4A90D9', legend=False)
ax1.set_title('Revenue by Product Category (purchases)')
ax1.set_xlabel('Revenue (USD)')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))

top_prod.sort_values('purchases', ascending=True).plot.barh(
    x='item_name', y='purchases', ax=ax2, color='#E67E22', legend=False)
ax2.set_title('Top 15 Products by Purchase Count')
ax2.set_xlabel('Purchases')
ax2.yaxis.set_tick_params(labelsize=9)

plt.tight_layout()
plt.show()

display(cat_rev[['item_category', 'distinct_products', 'revenue_usd', 'avg_price_usd']]
        .rename(columns={
            'item_category': 'Category', 'distinct_products': 'Products',
            'revenue_usd': 'Revenue (USD)', 'avg_price_usd': 'Avg Price (USD)'
        }))

In [ ]:
price_df = bq(f'''
SELECT item_category, price_in_usd
FROM `{RAW}.events_*`,
UNNEST(items)
WHERE _TABLE_SUFFIX BETWEEN '{D0}' AND '{D1}'
  AND event_name = 'view_item'
  AND item_category IN ('Apparel', 'Bags', 'Drinkware', 'Accessories', 'Office')
  AND price_in_usd > 0
  AND price_in_usd < 200
LIMIT 50000
''')

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=price_df, x='item_category', y='price_in_usd',
            ax=ax, palette='muted', showfliers=False)
ax.set_title('Price Distribution by Category (viewed items, outliers hidden)')
ax.set_xlabel('Category')
ax.set_ylabel('Price (USD)')
plt.tight_layout()
plt.show()

In [ ]:
aov_daily = bq(f'''
SELECT
  PARSE_DATE('%Y%m%d', event_date)                  AS date,
  ROUND(AVG(ecommerce.purchase_revenue_in_usd), 2)  AS aov_usd,
  COUNT(DISTINCT ecommerce.transaction_id)           AS transactions
FROM `{RAW}.events_*`
WHERE _TABLE_SUFFIX BETWEEN '{D0}' AND '{D1}'
  AND event_name = 'purchase'
  AND ecommerce.transaction_id IS NOT NULL
GROUP BY 1 ORDER BY 1
''')

overall_aov = aov_daily['aov_usd'].mean()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(aov_daily['date'], aov_daily['aov_usd'],
        color=C_CTRL, lw=1.5, alpha=0.7, label='Daily AOV')
ax.plot(aov_daily['date'],
        aov_daily['aov_usd'].rolling(7, center=True).mean(),
        color='#E63946', lw=2.5, label='7-day rolling avg')
ax.axhline(overall_aov, color='gray', linestyle='--', lw=1,
           label=f'Period avg: ${overall_aov:.0f}')
ax.set_title('Average Order Value Over Time')
ax.set_ylabel('AOV (USD)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Funnel Analysis

Step-over-step conversion, device heatmap, and drop-off rates.
Uses pre-aggregated `mart_funnel_metrics` (loaded in section 7).

In [ ]:
# Sum across all groups and devices to get overall funnel totals
funnel_totals = funnel_mart[FUNNEL_COLS].sum()
values = [int(funnel_totals[c]) for c in FUNNEL_COLS]

colors = sns.color_palette('Blues_r', len(values))
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(FUNNEL_LABELS[::-1], values[::-1], color=colors)
for bar, val, lbl in zip(bars, values[::-1], FUNNEL_LABELS[::-1]):
    pct = val / values[0] * 100
    ax.text(bar.get_width() + values[0] * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f'{val:,}  ({pct:.1f}%)', va='center', fontsize=10)
ax.set_title('Conversion Funnel — All Sessions (Nov 2020 – Jan 2021)')
ax.set_xlabel('Sessions')
ax.set_xlim(0, values[0] * 1.28)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

In [ ]:
rate_cols = ['view_item_rate_pct', 'add_to_cart_rate_pct',
             'begin_checkout_rate_pct', 'purchase_rate_pct',
             'overall_conversion_rate_pct']
rate_labels = [
    'View Item\n(% of sessions)',
    'Add to Cart\n(% of viewers)',
    'Begin Checkout\n(% of cart adds)',
    'Purchase\n(% of checkouts)',
    'Overall Conv.\n(session→purchase)',
]

device_rates = (
    funnel_mart.groupby('device_category')[rate_cols]
    .mean()
    .rename(columns=dict(zip(rate_cols, rate_labels)))
)

fig, ax = plt.subplots(figsize=(13, 3))
sns.heatmap(device_rates, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            linewidths=0.5, cbar_kws={'label': '%'})
ax.set_title('Funnel Conversion Rates by Device (%)')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
pairs = list(zip(FUNNEL_LABELS[:-1], FUNNEL_LABELS[1:], values[:-1], values[1:]))
drop_pct = [(a, b, (va - vb) / va * 100) for a, b, va, vb in pairs]
drop_df = pd.DataFrame(drop_pct, columns=['from', 'to', 'dropoff_pct'])
drop_df['label'] = drop_df['from'] + ' → ' + drop_df['to']

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(drop_df['label'], drop_df['dropoff_pct'],
              color='#E63946', edgecolor='white')
ax.set_title('Drop-off Rate at Each Funnel Step')
ax.set_ylabel('% of users lost at this step')
ax.set_ylim(0, 100)
for bar, val in zip(bars, drop_df['dropoff_pct'].tolist()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
            f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

## 10. A/A Test Exploration

Control vs Treatment — both groups saw the **same** website.
Expected result: **no significant difference** (z ≈ 0, p > 0.05).

In [ ]:
aa = bq(f'SELECT * FROM `{MART}.mart_aa_test_results`').iloc[0]
print(aa.to_string())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

# Group sizes
groups = ['Control', 'Treatment']
n_sess = [aa['ctrl_sessions'], aa['test_sessions']]
axes[0].bar(groups, n_sess, color=[C_CTRL, C_TRTM], edgecolor='white')
axes[0].set_title('Sessions per Group')
axes[0].set_ylabel('Sessions')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for bar, val in zip(axes[0].patches, n_sess):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
                 f'{val:,}', ha='center', fontsize=10)

# Conversion rates + 95% CI (Wilson approximation)
conv = [aa['ctrl_conversion_rate_pct'], aa['test_conversion_rate_pct']]
ns   = [aa['ctrl_sessions'], aa['test_sessions']]
ci95 = [1.96 * np.sqrt(r / 100 * (1 - r / 100) / n) * 100 for r, n in zip(conv, ns)]
axes[1].bar(groups, conv, color=[C_CTRL, C_TRTM], edgecolor='white',
            yerr=ci95, capsize=6, error_kw={'linewidth': 2})
axes[1].set_title('Conversion Rate ± 95% CI')
axes[1].set_ylabel('Purchase conv. rate (%)')
axes[1].set_ylim(0, max(conv) * 1.4)

# Revenue
rev = [aa['ctrl_revenue_usd'], aa['test_revenue_usd']]
axes[2].bar(groups, rev, color=[C_CTRL, C_TRTM], edgecolor='white')
axes[2].set_title('Total Revenue by Group')
axes[2].set_ylabel('Revenue (USD)')
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x):,}'))

plt.suptitle('A/A Test: Control vs Treatment (identical website)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Revenue per purchasing session — are distributions identical across groups?
purchase_rev = bq(f'''
SELECT a.aa_group, f.revenue_usd
FROM `{INT}.int_user_funnel` f
JOIN `{INT}.int_aa_test_groups` a USING (user_pseudo_id)
WHERE f.reached_purchase = 1
  AND f.revenue_usd IS NOT NULL
  AND f.revenue_usd > 0
''')

fig, ax = plt.subplots(figsize=(10, 4))
for group, color in [('control', C_CTRL), ('treatment', C_TRTM)]:
    subset = (purchase_rev
              .loc[purchase_rev['aa_group'] == group, 'revenue_usd']
              .clip(upper=500))
    subset.plot.kde(ax=ax, color=color, lw=2.5,
                    label=f'{group.title()} (n={len(subset):,})')
ax.set_title('Revenue per Purchasing Session — KDE (clipped at $500)')
ax.set_xlabel('Revenue (USD)')
ax.set_ylabel('Density')
ax.set_xlim(0, 500)
ax.legend()
plt.tight_layout()
plt.show()

for g in ['control', 'treatment']:
    vals = purchase_rev.loc[purchase_rev['aa_group'] == g, 'revenue_usd']
    print(f'{g.title():10s}  median=${vals.median():.2f}  mean=${vals.mean():.2f}')

In [ ]:
z = float(aa['z_score'])
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

x = np.linspace(-4, 4, 400)
y = stats.norm.pdf(x)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(x, y, color='#333', lw=2)
ax.fill_between(x, y, where=(x <= -1.96), color='#E63946', alpha=0.3,
                label='Rejection region (α=0.05)')
ax.fill_between(x, y, where=(x >= 1.96),  color='#E63946', alpha=0.3)
ax.fill_between(x, y, where=((x > -1.96) & (x < 1.96)),
                color='#4A90D9', alpha=0.15, label='Fail to reject H₀')
ax.axvline(z, color=C_TRTM, lw=2.5, linestyle='--',
           label=f'Our z = {z:.2f}  (p = {p_value:.3f})')
ax.axvline(-1.96, color='gray', lw=1, linestyle=':')
ax.axvline( 1.96, color='gray', lw=1, linestyle=':')
for xv in (-1.96, 1.96):
    ax.text(xv, max(y) * 0.97, f'{xv}', ha='center', fontsize=9, color='gray')
ax.set_title('Two-Proportion Z-Test: A/A Validation')
ax.set_xlabel('Z-score')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.show()

print(f'z-score      : {z:.4f}')
print(f'p-value      : {p_value:.4f}')
print(f'Sig. at 95%? : {abs(z) > 1.96}  ← expected False for a valid A/A test')

## 11. Key Takeaways

### Funnel
- **78% of sessions end before viewing a single product** — the biggest drop-off is at the very top of the funnel (homepage / navigation / search), not at checkout.
- Desktop converts at ~1.9%, mobile at ~0.7% — a 2.7× gap. Mobile checkout UX is the most impactful lever.
- 56% abandon between "Begin Checkout" and "Purchase" — payment friction or unexpected shipping costs are likely causes.

### Products & Revenue
- **Apparel** leads in both volume and revenue. **Drinkware** has high purchase frequency but a low average order value.
- AOV is noisy day-to-day but shows no strong seasonal trend within the 3-month window.

### A/A Test Validation
- z = -0.92, p ≈ 0.36 — **no statistically significant difference between groups**, as expected.
- Revenue distributions are visually indistinguishable. The `FARM_FINGERPRINT` split is unbiased and the pipeline is ready for real A/B experiments.

### Next steps for a real A/B test
1. Test a redesigned homepage or product listing page (largest drop-off point).
2. Prioritise mobile checkout flow improvements.
3. Use this pipeline's z-test infrastructure to evaluate any future experiment.